# 02  Data Preprocessing Pipeline
This notebook covers the full preprocessing pipeline for the retail customer churn dataset:
- Outlier detection (Isolation Forest)
- Feature engineering & encoding
- Dimensionality reduction (PCA)
- Clustering (KMeans)
- Train/test split & export

##  Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

##  Configuration

In [2]:
FILE_PATH = '../data/raw/retail_customers_COMPLETE_CATEGORICAL.csv'
OUTLIER_CONTAMINATION = 0.06
N_PCA_COMPONENTS = 10
N_CLUSTERS = 4
TEST_SIZE = 0.2
RANDOM_STATE = 42
HIGH_CARDINALITY_THRESHOLD = 0.90

##  Helper Function

In [3]:
def drop_high_cardinality(df, threshold=HIGH_CARDINALITY_THRESHOLD):
    """Drop columns with too many unique values (ID-like columns)."""
    cols_to_drop = []
    for col in df.columns:
        if col not in ['Churn', 'MonetaryTotal']:
            unique_ratio = df[col].nunique() / len(df)
            if unique_ratio > threshold:
                cols_to_drop.append(col)
    print(f"  Dropped high-cardinality columns: {cols_to_drop}")
    return df.drop(columns=cols_to_drop)

##  1. Load Data

In [4]:
assert os.path.exists(FILE_PATH), f"❌ File not found: {FILE_PATH}"

df = pd.read_csv(FILE_PATH)
print(f"✅ Loaded {len(df)} rows, {df.shape[1]} columns")
df.head()

✅ Loaded 4372 rows, 52 columns


,CustomerID,Recency,Frequency,MonetaryTotal,MonetaryAvg,MonetaryStd,MonetaryMin,MonetaryMax,TotalQuantity,AvgQuantityPerTransaction,...,Region,LoyaltyLevel,ChurnRiskCategory,WeekendPreference,BasketSizeCategory,ProductDiversity,Gender,AccountStatus,Country,Churn
0,17850,302,35,5288.63,16.950737,13.603662,-30.60,107.25,1693,5.426282,...,UK,Jeune,Critique,Inconnu,Moyen,Explorateur,Unknown,Active,United Kingdom,1
1,13047,32,18,3079.10,15.709694,11.684769,-15.00,68.00,1355,6.913265,...,UK,Établi,Moyen,Semaine,Moyen,Explorateur,M,Active,United Kingdom,0
2,12583,3,18,7187.34,28.634821,23.150132,-60.84,132.80,5009,19.956175,...,Europe continentale,Ancien,Faible,Semaine,Grand,Explorateur,Unknown,Active,France,0
3,13748,96,5,948.25,33.866071,42.953119,9.36,204.00,439,15.678571,...,UK,Établi,Critique,Inconnu,Grand,Explorateur,Unknown,Active,United Kingdom,1
4,15100,330,6,635.10,105.850000,215.986263,-131.40,350.40,58,9.666667,...,UK,Jeune,Critique,Inconnu,Moyen,Spécialisé,M,Active,United Kingdom,1


In [5]:
# Keep a raw backup for later use
df_raw_backup = df.copy()
print("Raw backup saved.")

Raw backup saved.


##  2. Outlier Detection (Isolation Forest) & Date Parsing

In [6]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
features_for_outlier = [c for c in numeric_cols if c not in ['Churn', 'CustomerID']]

iso_forest = IsolationForest(contamination=OUTLIER_CONTAMINATION, random_state=RANDOM_STATE)
outlier_preds = iso_forest.fit_predict(df[features_for_outlier].fillna(0))

n_before = len(df)
df = df[outlier_preds == 1].reset_index(drop=True)
print(f"✅ Outliers removed: {n_before - len(df)} rows dropped → {len(df)} rows remaining")

✅ Outliers removed: 263 rows dropped → 4109 rows remaining


In [7]:
# Parse registration date and extract year/month features
df['RegistrationDate'] = pd.to_datetime(df['RegistrationDate'], dayfirst=True, errors='coerce')
df['RegYear']  = df['RegistrationDate'].dt.year.fillna(df['RegistrationDate'].dt.year.median())
df['RegMonth'] = df['RegistrationDate'].dt.month.fillna(df['RegistrationDate'].dt.month.median())

print("✅ Date features extracted: RegYear, RegMonth")
df[['RegistrationDate', 'RegYear', 'RegMonth']].head()

✅ Date features extracted: RegYear, RegMonth


,RegistrationDate,RegYear,RegMonth
0,2010-04-10,2010.0,4.0
1,NaT,2011.0,7.0
2,NaT,2011.0,7.0
3,NaT,2011.0,7.0
4,NaT,2011.0,7.0


##  3. Cleaning & Encoding

In [8]:
# Save the regression target before dropping columns
y_reg_full = df['MonetaryTotal'].fillna(df['MonetaryTotal'].median())
print(f"✅ Regression target (MonetaryTotal) saved — median: {y_reg_full.median():.2f}")

✅ Regression target (MonetaryTotal) saved — median: 609.19


In [9]:
cols_to_drop = [
    'Recency', 'AccountStatus', 'RFMSegment', 'ChurnRiskCategory',
    'CustomerID', 'RegistrationDate', 'LastLoginIP', 'NewsletterSubscribed',
    'MonetaryAvg', 'TotalQuantity', 'TotalTransactions'
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
df = drop_high_cardinality(df)

print(f"✅ After column drops: {df.shape[1]} columns remaining")

  Dropped high-cardinality columns: ['MonetaryStd']
✅ After column drops: 42 columns remaining


In [10]:
# Label-encode categorical columns (excluding 'Churn')
cat_cols = [c for c in df.select_dtypes(include=['object']).columns if c != 'Churn']
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

print(f"✅ Encoded {len(cat_cols)} categorical columns: {cat_cols}")

# Fill remaining NaNs with column medians
df = df.fillna(df.median())
print(f"✅ NaN values filled with column medians. Remaining NaNs: {df.isna().sum().sum()}")

✅ Encoded 12 categorical columns: ['AgeCategory', 'SpendingCategory', 'CustomerType', 'FavoriteSeason', 'PreferredTimeOfDay', 'Region', 'LoyaltyLevel', 'WeekendPreference', 'BasketSizeCategory', 'ProductDiversity', 'Gender', 'Country']
✅ NaN values filled with column medians. Remaining NaNs: 0


##  4. PCA — Dimensionality Reduction

In [11]:
X_raw   = df.drop(columns=['Churn', 'MonetaryTotal'], errors='ignore')
y_class = df['Churn']

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Store expected column names for raw export later
colonnes_attendues = scaler.feature_names_in_.tolist()

print(f"✅ Features scaled: {X_scaled.shape}")

✅ Features scaled: (4109, 40)


In [12]:
pca   = PCA(n_components=N_PCA_COMPONENTS, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

X_final = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(N_PCA_COMPONENTS)])

explained = pca.explained_variance_ratio_.cumsum()
print(f"✅ PCA complete: {N_PCA_COMPONENTS} components explain {explained[-1]*100:.1f}% of variance")
X_final.head()

✅ PCA complete: 10 components explain 58.5% of variance


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,9.047337,-3.235069,-0.888405,4.978797,-1.508251,1.725227,1.284199,-0.303970,1.975817,0.244387
1,5.243957,-2.414492,-0.218542,-2.256103,-4.960781,-1.340014,0.287484,-0.789458,0.219696,1.127308
2,0.261418,-2.231353,-0.256878,-1.368653,0.753170,0.809285,-1.632114,-1.634942,-1.509161,0.909652
3,5.387821,-3.422244,1.037239,0.011424,-1.124219,1.507662,-2.278130,0.683456,1.381382,-1.579936
4,5.418090,-3.437700,-0.455061,0.849134,-2.793211,1.636965,-1.391206,1.463831,-4.059282,-0.391255


##  5. KMeans Clustering on PCA Components

In [13]:
print(f"Training KMeans with {N_CLUSTERS} clusters on {N_PCA_COMPONENTS} PCA components...")
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(X_final)

cluster_counts = pd.Series(kmeans.labels_).value_counts().sort_index()
print("✅ KMeans trained. Cluster distribution:")
print(cluster_counts.to_string())

Training KMeans with 4 clusters on 10 PCA components...
✅ KMeans trained. Cluster distribution:
0     757
1    1878
2     479
3     995


##  6. Save Processed Data

In [14]:
os.makedirs('../data/processed', exist_ok=True)

df_processed_all = X_final.copy()
df_processed_all['Churn']         = y_class.values
df_processed_all['MonetaryTotal'] = y_reg_full.values

df_processed_all.to_csv('../data/processed/processed_data_final.csv', index=False)
print(f"✅ Saved: ../data/processed/processed_data_final.csv — shape: {df_processed_all.shape}")

✅ Saved: ../data/processed/processed_data_final.csv — shape: (4109, 12)


##  7. Train / Test Split & Export

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_class,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_class
)

print(f"✅ Split complete:")
print(f"   Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")
print(f"   Churn rate — Train: {y_train.mean():.2%} | Test: {y_test.mean():.2%}")

✅ Split complete:
   Train: 3287 samples | Test: 822 samples
   Churn rate — Train: 34.38% | Test: 34.43%


In [16]:
os.makedirs('../data/train_test', exist_ok=True)
os.makedirs('../models', exist_ok=True)

X_train.to_csv('../data/train_test/X_train.csv', index=False)
X_test.to_csv('../data/train_test/X_test.csv',   index=False)
y_train.to_csv('../data/train_test/y_train.csv', index=False)
y_test.to_csv('../data/train_test/y_test.csv',   index=False)

joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(pca,    '../models/pca_model.pkl')
joblib.dump(kmeans, '../models/kmeans_model.pkl')

print("✅ Saved train/test CSVs and model artifacts (scaler, PCA, KMeans).")

✅ Saved train/test CSVs and model artifacts (scaler, PCA, KMeans).


##  8. Generate Raw Test File (`X_test_brut_40.csv`)

In [17]:
# Retrieve the original (pre-PCA) feature values for test rows
# df has already been cleaned/encoded and shares the same index as X_final
df_test_brut = df.loc[X_test.index, colonnes_attendues]

output_brut_path = '../data/train_test/X_test_brut_40.csv'
df_test_brut.to_csv(output_brut_path, index=False)

print(f"✅ Saved: {output_brut_path}")
print(f"   Shape: {df_test_brut.shape} (expected {len(colonnes_attendues)} columns)")
df_test_brut.head()

✅ Saved: ../data/train_test/X_test_brut_40.csv
   Shape: (822, 40) (expected 40 columns)


,Frequency,MonetaryMin,MonetaryMax,AvgQuantityPerTransaction,MinQuantity,MaxQuantity,CustomerTenureDays,FirstPurchaseDaysAgo,PreferredDayOfWeek,PreferredHour,...,PreferredTimeOfDay,Region,LoyaltyLevel,WeekendPreference,BasketSizeCategory,ProductDiversity,Gender,Country,RegYear,RegMonth
1985,1,6.96,19.8,11.000000,2,24,0,253,4,10,...,1,12,2,0,1,1,2,34,2011.0,7.0
1775,7,-21.90,70.8,9.367347,-2,24,250,268,0,12,...,2,12,3,0,1,0,0,34,2011.0,7.0
919,4,5.04,40.8,9.108696,1,48,261,332,1,13,...,2,6,3,1,1,0,2,13,2011.0,7.0
1272,2,4.56,131.4,34.000000,12,144,89,309,3,9,...,1,12,1,0,0,1,0,34,2011.0,7.0
1401,8,-0.42,237.6,32.120000,-1,96,285,295,4,14,...,0,12,3,1,0,0,2,34,2011.0,7.0


##  Summary

In [18]:
print("=" * 70)
print(" " * 15 + "PREPROCESSING PIPELINE — COMPLETE")
print("=" * 70)
print("\n DATA SUMMARY:")
print(f"  • Input rows (after outlier removal)   : {len(df):,}")
print(f"  • Original features (before PCA)       : {len(colonnes_attendues)}")
print(f"  • PCA components (final features)      : {N_PCA_COMPONENTS}")
print(f"  • Variance explained by PCA            : {explained[-1]*100:.1f}%")
print(f"  • Dimensionality reduction ratio       : {len(colonnes_attendues)} → {N_PCA_COMPONENTS} ({N_PCA_COMPONENTS/len(colonnes_attendues)*100:.1f}%)")

print("\n TRAIN / TEST SPLIT:")
print(f"  • Split ratio                         : {1-TEST_SIZE:.0%} train / {TEST_SIZE:.0%} test")
print(f"  • Train samples                       : {len(X_train):,}")
print(f"  • Test samples                        : {len(X_test):,}")
print(f"  • Churn rate (Train)                  : {y_train.mean():.2%}")
print(f"  • Churn rate (Test)                   : {y_test.mean():.2%}")

print("\n  CLUSTERING:")
print(f"  • KMeans clusters                     : {N_CLUSTERS}")
print(f"  • Cluster distribution:")
for cluster_id, count in cluster_counts.items():
    print(f"      - Cluster {cluster_id}: {count:,} samples ({count/len(df)*100:.1f}%)")

print("\n SAVED ARTIFACTS:")
artifacts = {
    "../models/scaler.pkl": "StandardScaler model",
    "../models/pca_model.pkl": "PCA transformation model",
    "../models/kmeans_model.pkl": "KMeans clustering model",
    "../data/processed/processed_data_final.csv": f"Full processed data ({df_processed_all.shape[0]:,} rows × {df_processed_all.shape[1]} cols)",
    "../data/train_test/X_train.csv": f"Training features ({X_train.shape[0]:,} rows × {X_train.shape[1]} cols)",
    "../data/train_test/X_test.csv": f"Test features ({X_test.shape[0]:,} rows × {X_test.shape[1]} cols)",
    "../data/train_test/y_train.csv": f"Training target ({len(y_train):,} rows)",
    "../data/train_test/y_test.csv": f"Test target ({len(y_test):,} rows)",
    "../data/train_test/X_test_brut_40.csv": f"Raw test features ({df_test_brut.shape[0]:,} rows × {df_test_brut.shape[1]} cols)"
}

for path, description in artifacts.items():
    exists = "✅" if os.path.exists(path) else "❌"
    print(f"  {exists} {path:<50} ({description})")

print("\n" + "=" * 70)

               PREPROCESSING PIPELINE — COMPLETE

 DATA SUMMARY:
  • Input rows (after outlier removal)   : 4,109
  • Original features (before PCA)       : 40
  • PCA components (final features)      : 10
  • Variance explained by PCA            : 58.5%
  • Dimensionality reduction ratio       : 40 → 10 (25.0%)

 TRAIN / TEST SPLIT:
  • Split ratio                         : 80% train / 20% test
  • Train samples                       : 3,287
  • Test samples                        : 822
  • Churn rate (Train)                  : 34.38%
  • Churn rate (Test)                   : 34.43%

  CLUSTERING:
  • KMeans clusters                     : 4
  • Cluster distribution:
      - Cluster 0: 757 samples (18.4%)
      - Cluster 1: 1,878 samples (45.7%)
      - Cluster 2: 479 samples (11.7%)
      - Cluster 3: 995 samples (24.2%)

 SAVED ARTIFACTS:
  ✅ ../models/scaler.pkl                               (StandardScaler model)
  ✅ ../models/pca_model.pkl                            (PCA transform